# Sentiment Analysis of Amazon Product Reviews

In this project I analyse Amazon product reviews to understand customer sentiment.
I use Python for cleaning and analysis, export the data to Excel where I apply text functions and create a pivot table, and then train a Naive Bayes model to predict whether a review is positive or negative.

**Dataset:** Amazon India product reviews (downloaded from course materials)


## Step 1 – Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## Step 2 – Load the data

The CSV has one row per product with multiple reviews packed into a single cell separated by commas.
I need to split them out so each row is one review.


In [ ]:
df_raw = pd.read_csv('amazon.csv')
print('Shape:', df_raw.shape)
df_raw.head(3)

In [ ]:
rows = []

for _, row in df_raw.iterrows():
    titles   = [t.strip() for t in str(row['review_title']).split(',')]
    contents = [c.strip() for c in str(row['review_content']).split(',')]
    n = max(len(titles), len(contents))

    for i in range(n):
        rows.append({
            'product' : str(row['product_name'])[:60],
            'category': str(row['category']).split('|')[0],
            'rating'  : str(row['rating']),
            'title'   : titles[i]   if i < len(titles)   else '',
            'review'  : contents[i] if i < len(contents) else '',
        })

df = pd.DataFrame(rows)
print('Total reviews:', len(df))
df.head(5)

## Step 3 – Clean the data

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Convert rating from string to float
def parse_rating(r):
    try:
        val = float(str(r).strip())
        if 1 <= val <= 5:
            return val
    except:
        pass
    return None

df['rating'] = df['rating'].apply(parse_rating)
df = df.dropna(subset=['rating'])
print('Rows after cleaning:', len(df))

In [ ]:
# Combine title and review text
df['review_text'] = (df['title'] + ' ' + df['review']).str.strip()
df = df[df['review_text'].str.len() > 5].reset_index(drop=True)

# Add sentiment label
def get_sentiment(r):
    if r >= 4:
        return 'Positive'
    elif r <= 2:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment']      = df['rating'].apply(get_sentiment)
df['review_length']  = df['review_text'].apply(len)

print(df['sentiment'].value_counts())
df.head(5)

## Step 4 – Export to Excel

I export the cleaned data to Excel. The file will have two sheets:
- **Reviews** – the cleaned dataset (I will add formulas to this manually in Excel)
- **Word Frequency** – frequency count of positive vs negative words


In [ ]:
positive_words = [
    'good', 'great', 'excellent', 'amazing', 'perfect', 'love', 'best',
    'fantastic', 'awesome', 'quality', 'happy', 'recommend', 'satisfied',
    'wonderful', 'nice', 'easy', 'fast', 'reliable', 'value', 'superb'
]

negative_words = [
    'bad', 'terrible', 'awful', 'worst', 'horrible', 'broke', 'broken',
    'waste', 'poor', 'disappointed', 'slow', 'cheap', 'fake', 'useless',
    'damaged', 'defective', 'problem', 'issue', 'failed', 'return'
]

# Count how many reviews contain each word
pos_freq = {w: int(df['review_text'].str.lower().str.contains(w).sum()) for w in positive_words}
neg_freq = {w: int(df['review_text'].str.lower().str.contains(w).sum()) for w in negative_words}

pos_df = pd.DataFrame(list(pos_freq.items()), columns=['word', 'count']).sort_values('count', ascending=False)
neg_df = pd.DataFrame(list(neg_freq.items()), columns=['word', 'count']).sort_values('count', ascending=False)
pos_df['type'] = 'Positive'
neg_df['type'] = 'Negative'

word_freq_df = pd.concat([pos_df, neg_df], ignore_index=True)

print('Word frequency table:')
print(word_freq_df.head(10))

In [ ]:
# Export to Excel
excel_path = 'amazon_reviews.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # Sheet 1: Reviews (first 1000 rows so the file stays manageable)
    df[['product', 'category', 'rating', 'sentiment', 'review_text', 'review_length']].head(1000).to_excel(
        writer, sheet_name='Reviews', index=False
    )

    # Sheet 2: Word frequency count
    word_freq_df.to_excel(writer, sheet_name='Word Frequency', index=False)

print(f'Saved: {excel_path}')

## Step 5 – Excel: text functions and pivot table

After running this notebook, open **amazon_reviews.xlsx** in Excel and follow the steps below.

---

### Part A – Text functions on the Reviews sheet

The data is in columns A–F. Column E contains the review text.

Add these formulas in new columns to the right:

**Column G — review character count using LEN:**
```
=LEN(E2)
```
This counts how many characters are in the review text.

**Column H — find the word "good" using FIND:**
```
=IFERROR(FIND("good", LOWER(E2)), "not found")
```
This returns the position of the word "good" in the review. If it's not there it returns "not found".

**Column I — remove the stop word "the" using SUBSTITUTE:**
```
=SUBSTITUTE(LOWER(E2), "the ", "")
```
This removes the word "the" from the review text. You can chain multiple SUBSTITUTE calls to remove more stop words:
```
=SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(LOWER(E2), "the ", ""), "and ", ""), "is ", "")
```

**Column J — count how many times "good" appears using LEN and SUBSTITUTE:**
```
=(LEN(LOWER(E2)) - LEN(SUBSTITUTE(LOWER(E2), "good", ""))) / LEN("good")
```
This is the standard Excel trick: subtract the length after removing the word from the original length, then divide by the word length.

After typing the formula in row 2, drag it down for all rows.

---

### Part B – Pivot table on the Reviews sheet

1. Click anywhere inside the data on the Reviews sheet
2. Go to **Insert → PivotTable**
3. Select "New Worksheet" and click OK
4. In the PivotTable Fields panel:
   - Drag **category** to the **Rows** area
   - Drag **sentiment** to the **Columns** area
   - Drag **rating** to the **Values** area, set it to **Count**

This gives you a table showing how many Positive, Negative, and Neutral reviews each category has.

You can also make a second pivot table with **product** in Rows to see sentiment per product.

---

The screenshots below show what the final Excel file looks like after completing these steps.


## Step 6 – Exploratory Data Analysis

In [ ]:
# Rating distribution
plt.figure(figsize=(8, 4))
df['rating'].round().value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Number of Reviews by Star Rating')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Sentiment breakdown
counts = df['sentiment'].value_counts()

plt.figure(figsize=(6, 5))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%',
        colors=['#4CAF50', '#FFC107', '#F44336'], startangle=90)
plt.title('Sentiment Distribution')
plt.show()
print(counts)

In [ ]:
# Average rating by category
plt.figure(figsize=(10, 4))
df.groupby('category')['rating'].mean().sort_values(ascending=False).plot(
    kind='bar', color='steelblue', edgecolor='black'
)
plt.title('Average Rating by Category')
plt.xlabel('')
plt.ylabel('Average Rating')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Positive vs negative word frequency
pos_sorted = sorted(pos_freq.items(), key=lambda x: x[1], reverse=True)[:10]
neg_sorted = sorted(neg_freq.items(), key=lambda x: x[1], reverse=True)[:10]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([w for w, _ in pos_sorted][::-1], [c for _, c in pos_sorted][::-1], color='#4CAF50')
axes[0].set_title('Top Positive Words')
axes[0].set_xlabel('Reviews containing word')

axes[1].barh([w for w, _ in neg_sorted][::-1], [c for _, c in neg_sorted][::-1], color='#F44336')
axes[1].set_title('Top Negative Words')
axes[1].set_xlabel('Reviews containing word')

plt.tight_layout()
plt.show()

In [ ]:
# Sentiment by category
pivot = df.groupby(['category', 'sentiment']).size().unstack(fill_value=0)

pivot.plot(kind='bar', figsize=(12, 5), color=['#F44336', '#FFC107', '#4CAF50'], edgecolor='black')
plt.title('Sentiment Count by Category')
plt.xlabel('')
plt.ylabel('Number of reviews')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

## Step 7 – Machine Learning: Naive Bayes

I train a Naive Bayes model to predict whether a review is **Positive** or **Negative** based on the words in it.

**Why Naive Bayes?**
It works well for text classification. It learns which words appear most often in positive vs negative reviews and uses that pattern to classify new reviews.


In [ ]:
# Keep only Positive and Negative reviews
df_ml = df[df['sentiment'].isin(['Positive', 'Negative'])].copy()
print(df_ml['sentiment'].value_counts())

In [ ]:
# Balance the classes - sample equal numbers of positive and negative
n = (df_ml['sentiment'] == 'Negative').sum()

df_pos = df_ml[df_ml['sentiment'] == 'Positive'].sample(n, random_state=42)
df_neg = df_ml[df_ml['sentiment'] == 'Negative']

df_balanced = pd.concat([df_pos, df_neg]).reset_index(drop=True)
print('After balancing:')
print(df_balanced['sentiment'].value_counts())

In [ ]:
# Train / test split
X = df_balanced['review_text']
y = df_balanced['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training samples:', len(X_train))
print('Test samples    :', len(X_test))

In [ ]:
# Convert text to word count vectors
vectorizer = CountVectorizer(stop_words='english', max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# Train the model
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Predict on the test set
y_pred = model.predict(X_test_vec)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 2))
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Negative'])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Negative'],
            yticklabels=['Positive', 'Negative'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Compare predictions with actual ratings
# This shows the model's prediction next to the actual star rating

results = pd.DataFrame({
    'review'          : X_test.str[:70].values + '...',
    'actual_rating'   : df_balanced.loc[X_test.index, 'rating'].values,
    'actual_label'    : y_test.values,
    'predicted_label' : y_pred,
})
results['correct'] = results['actual_label'] == results['predicted_label']

print('Sample of predictions vs actual ratings:')
results.head(20)

In [ ]:
# Accuracy per class
for label in ['Positive', 'Negative']:
    subset = results[results['actual_label'] == label]
    print(f'{label}: {subset["correct"].mean():.1%} correct ({len(subset)} reviews)')

## Summary

**What I did:**
1. Loaded the Amazon reviews dataset and expanded product rows into individual reviews
2. Cleaned the data — parsed ratings, removed empty rows, labelled sentiment
3. Exported to Excel and manually applied LEN, FIND, SUBSTITUTE formulas and created a pivot table
4. Explored the data with charts
5. Trained a Naive Bayes model to classify reviews as positive or negative and compared predictions against actual star ratings

**Findings:**
- Most reviews are 4–5 stars — Amazon reviews skew positive
- "good", "great", and "quality" are the most common positive words
- "bad", "poor", and "broke" are the most common negative words  
- The Naive Bayes model can predict positive vs negative reviews with reasonable accuracy based on word frequency alone
